In [1]:
import json
import os
import cv2
from glob import glob

INPUT_IMAGES_DIR = 'unity/images'
INPUT_LABELS_DIR = 'unity/labels'

OUTPUT_YOLO_LABELS = 'lpr_datasets/yolo/labels'
OUTPUT_LPR_CROPS = 'lpr_datasets/lprnet/crops'

os.makedirs(OUTPUT_YOLO_LABELS, exist_ok=True)
os.makedirs(OUTPUT_LPR_CROPS, exist_ok=True)

def get_merged_bbox(char_objects):
    x1_vals = [c['bbox'][0] for c in char_objects]
    y1_vals = [c['bbox'][1] for c in char_objects]
    x2_vals = [c['bbox'][2] for c in char_objects]
    y2_vals = [c['bbox'][3] for c in char_objects]

    min_x = min(x1_vals)
    min_y = min(y1_vals)
    max_x = max(x2_vals)
    max_y = max(y2_vals)

    return [min_x, min_y, max_x, max_y]

def is_number(text):
    allowed = set("0123456789.-")
    return all(c in allowed for c in text) and len(text) > 0

def process_files():
    json_files = glob(os.path.join(INPUT_LABELS_DIR, '*.json'))

    print(f"Знайдено {len(json_files)} файлів для обробки...")

    for json_path in json_files:
        with open(json_path, 'r', encoding='utf-8') as f:
            data_list = json.load(f)
        base_name = os.path.basename(json_path).replace('.json', '')
        img_path = os.path.join(INPUT_IMAGES_DIR, base_name + '.jpg')
        if not os.path.exists(img_path):
             img_path = os.path.join(INPUT_IMAGES_DIR, base_name + '.png')

        if not os.path.exists(img_path):
            print(f"Увага: Картинку для {base_name} не знайдено, пропускаємо.")
            continue

        img = cv2.imread(img_path)
        h_img, w_img = img.shape[:2]
        yolo_lines = []

        for item in data_list:
            chars_pool = item['chars']
            data_raw_groups = item['data_raw']

            char_cursor = 0

            for group in data_raw_groups:
                for text_word in group:
                    word_len = len(text_word)
                    word_chars = chars_pool[char_cursor : char_cursor + word_len]
                    char_cursor += word_len

                    if is_number(text_word):
                        bbox = get_merged_bbox(word_chars)
                        pad = 2
                        x1_c = max(0, int(bbox[0]) - pad)
                        y1_c = max(0, int(bbox[1]) - pad)
                        x2_c = min(w_img, int(bbox[2]) + pad)
                        y2_c = min(h_img, int(bbox[3]) + pad)

                        crop = img[y1_c:y2_c, x1_c:x2_c]
                        save_name = f"{base_name}_{text_word}.jpg"
                        cv2.imwrite(os.path.join(OUTPUT_LPR_CROPS, save_name), crop)

                        bw = (bbox[2] - bbox[0])
                        bh = (bbox[3] - bbox[1])
                        bx_center = bbox[0] + bw / 2
                        by_center = bbox[1] + bh / 2

                        xn = bx_center / w_img
                        yn = by_center / h_img
                        wn = bw / w_img
                        hn = bh / h_img

                        yolo_lines.append(f"0 {xn:.6f} {yn:.6f} {wn:.6f} {hn:.6f}")

        if yolo_lines:
            yolo_txt_path = os.path.join(OUTPUT_YOLO_LABELS, base_name + '.txt')
            with open(yolo_txt_path, 'w') as f:
                f.write("\n".join(yolo_lines))

    print("Готово! Перевірте папки datasets/")

if __name__ == "__main__":
    process_files()

Знайдено 6637 файлів для обробки...
Готово! Перевірте папки datasets/
